# Projects in Math Modelling - Part 1 Sentiment Analysis

# Web Scrapping

https://pypi.org/project/twikit/
https://blog.apify.com/how-to-scrape-tweets-and-more-on-twitter-59330e6fb522/

In [13]:
pip install twikit


[notice] A new release of pip is available: 24.1 -> 24.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
#install packages
from twikit import Client ##for web scraping
import json
import pandas as pd
import time

In [15]:
#Load starter list

#path="C:/Users/leaka/Documents/Uni/Master/_Projects in Math Modelling/Part2/code/"
#athletes=pd.read_csv(path+'5000m_all_time_toplist.csv')
with open('starterlist_final.json', 'r') as json_file:
    athletes = json.load(json_file)

#Take competitor column and format it into a string list to use later on as keywords
#competitor = athletes["Competitor"].astype(str).tolist()
competitor = [name.title() for name in athletes]

In [16]:
#initialize client
client = Client('en-IE')

In [17]:
#Pull Twitter login information
with open("twitter_login.json") as infile:
    json_obj = json.load(infile)
    username =json_obj["E-Mail4"]
    password =json_obj["Password4"]

In [18]:
#pip install asyncio

In [19]:
#Loggin into account on Twitter
client.login(auth_info_1=username, password=password)
client.save_cookies('cookies.json')
client.load_cookies(path='cookies.json')

/var/folders/4g/5jg5kwgs5l70tgz45sxyy4gr0000gn/T/ipykernel_17350/2253358876.py:2: RuntimeWarning: coroutine 'Client.login' was never awaited
  client.login(auth_info_1=username, password=password)


In [20]:
# Function to handle rate limit (only limited amount of requests per 15 minutes allowed)
def handle_rate_limit():
    print("Rate limit hit. Sleeping for 15 minutes.")
    time.sleep(15 * 60)  # Sleep for 15 minutes

In [28]:
#webscape the tweets
tweets_to_store = []
for name in competitor:
    query = f'"{name}"'
    
    while True:
        try:
            tweets = client.search_tweet(query, 'Latest')
            break  # Exit the loop if request was successful
        except Exception as e:
            if "Rate limit exceeded" in str(e):
                handle_rate_limit()
            else:
                print(f"An error occurred: {e}")
                break


    if tweets is None:
        print(f"No tweets found for athlete: {name}")
        continue
    
    for tweet in tweets:
        tweets_to_store.append({
            'created_at': tweet.created_at,
            'username': tweet.user.name,
            'Text': tweet.text,
            'Competitor': name
        })
    
    # Sleep for a short interval to avoid hitting the rate limit
    time.sleep(5)

/var/folders/4g/5jg5kwgs5l70tgz45sxyy4gr0000gn/T/ipykernel_17350/2209243422.py:8: RuntimeWarning: coroutine 'Client.search_tweet' was never awaited
  tweets = client.search_tweet(query, 'Latest')


TypeError: 'coroutine' object is not iterable

In [ ]:
# Save the tweet details to a JSON file
with open('tweets_prerace.json', 'w') as json_file:
    json.dump(tweets_to_store, json_file, indent=4)


